# Exploración de la capa Gold

Notebook de análisis exploratorio para validar los insights del resumen ejecutivo.
Lee directamente del warehouse DuckDB construido por el pipeline.

**Pre-requisito:** haber corrido `python main.py` antes.

In [ ]:
import duckdb
import pandas as pd
import plotly.express as px
from pathlib import Path

DUCKDB_PATH = Path('..') / 'data' / 'gold' / 'warehouse.duckdb'
con = duckdb.connect(str(DUCKDB_PATH), read_only=True)
print('Tablas en el warehouse:')
print(con.execute('SHOW TABLES').fetchdf())

## 1. KPIs globales

In [ ]:
con.execute('SELECT * FROM gold_kpi_overall').fetchdf().T

## 2. Por método de pago

In [ ]:
by_pm = con.execute('SELECT * FROM gold_kpi_by_payment_method').fetchdf()
by_pm

In [ ]:
fig = px.bar(by_pm, x='payment_method', y='total_amount', color='payment_method',
             title='Volumen económico por método de pago')
fig.show()

## 3. Cruce método × canal (mapa de fricción)

In [ ]:
mc = con.execute('SELECT * FROM gold_kpi_by_method_channel ORDER BY failure_rate DESC').fetchdf()
mc.head(10)

In [ ]:
pivot = mc.pivot(index='payment_method', columns='channel', values='success_rate')
fig = px.imshow(pivot, text_auto='.1%', color_continuous_scale='RdYlGn',
                title='Tasa de éxito: método × canal')
fig.show()

## 4. Tendencia mensual

In [ ]:
monthly = con.execute('''
    SELECT date_trunc('month', transaction_date)::DATE AS mes,
           SUM(total_transactions) AS tx,
           AVG(success_rate) AS success_rate,
           SUM(total_amount) AS amount
    FROM gold_kpi_by_day
    GROUP BY 1
    ORDER BY 1
''').fetchdf()
fig = px.line(monthly, x='mes', y='success_rate', markers=True,
              title='Tasa de éxito mensual')
fig.update_yaxes(tickformat='.1%')
fig.show()

## 5. Top 10 usuarios por revenue

In [ ]:
top = con.execute('SELECT user_id, user_name, total_transactions, revenue, success_rate FROM gold_kpi_by_user ORDER BY revenue DESC LIMIT 10').fetchdf()
top

In [ ]:
con.close()